# From structural equations to causal discovery

In this tutorial, we use a simple linear simulator to generate observational and interventional data from a DAG under an additive-noise linear structural causal model (SCM). We then use CORNETO's `LinearDAGDiscovery` method to see whether we can recover the generating DAG.

By the end, you will be able to define a simple linear SCM, simulate observations and hard interventions, and discover a directed acyclic graph (DAG) from a complete candidate network.

This notebook follows CORNETO's [SEM and SCM simulation guide](https://corneto.org/stable/guide/causal/scm-simulation.html) and [linear DAG discovery guide](https://corneto.org/stable/guide/causal/linear-dag-discovery.html). Run the cells in order using the course environment (`pixi run notebook`).

> **Note:** This method is designed to discover causal pathways downstream of interventions. It selects edges only when they lie on a directed path from an intervened variable to a measured response. It may therefore omit upstream causes when an upstream variable is never intervened on. An omitted edge does not necessarily imply the absence of a causal effect.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import corneto as cn
from corneto.data import Data
from corneto.methods import LinearDAGDiscovery
from corneto.methods.causal.simulation import Normal, linear_scm
from utils import to_df, from_df, plot_dag_solution, plot_dag_errors, plot_dag_predictions

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

cn.info()

## 1. Specify the generating model

Our ground truth is **C ← A → B, with C → B**. A influences B directly and indirectly through C:

$$
A = \varepsilon_A,\qquad
C = 0.8A + \varepsilon_C,\qquad
B = 1.4A + 1.2C + \varepsilon_B.
$$

We draw independent, zero-mean Gaussian noise:

$$
\varepsilon_A \sim \mathcal{N}(0, 1),\qquad
\varepsilon_C \sim \mathcal{N}(0, 2.25),\qquad
\varepsilon_B \sim \mathcal{N}(0, 0.25).
$$

Here the second parameter is the **variance**, so the standard deviations are 1.0 for A, 1.5 for C, and 0.5 for B. A is a **root**: it has no parents. The noise represents unexplained variation between samples.

An SEM specifies equations. Here, we interpret them as causal mechanisms, making this an SCM: manipulating one variable changes its mechanism while preserving the others. We assume that all relevant variables are measured and that there are no hidden common causes.

In [ ]:
weights = {("A", "C"): 0.8, ("A", "B"): 1.4, ("C", "B"): 1.2}
truth_graph = cn.Graph.from_tuples([(u, 1, v) for u, v in weights])
truth = linear_scm(
    truth_graph,
    weights=weights,
    noise={"A": Normal(std=1.0), "C": Normal(std=1.5), "B": Normal(std=0.5)},
)
variables = ["A", "B", "C"]
truth_graph.plot(
    custom_edge_attr={i: {"label": f"{weights[(next(iter(u)), next(iter(v)))]:.1f}"}
                      for i, (u, v) in enumerate(truth_graph.E)},
)

In [ ]:
truth.variables

## 2. Sample data from the system

Each row is an independent sample. `sample()` returns columns in `truth.variables` order, so we explicitly label them before reordering for display.

**Before running:** which pairs do you expect to be correlated? Does a correlation tell us which way the arrow points?

In [ ]:
observations = pd.DataFrame(truth.sample(500, rng=7), columns=truth.variables)[variables]
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, (u, v) in zip(axes, [("A", "C"), ("A", "B"), ("C", "B")]):
    ax.scatter(observations[u], observations[v], s=10, alpha=0.4)
    ax.set(xlabel=u, ylabel=v)
fig.suptitle("Observational associations")
fig.tight_layout()

In [ ]:
observations.corr().round(2)

All three pairs are associated. This fully connected three-node DAG has no conditional independences to distinguish it from the other fully connected acyclic orientations. Without extra assumptions, linear Gaussian observational data alone cannot identify the causal ordering.

We will add experiments that intervene on each variable in the model.

## 3. Intervene on every variable

A **hard intervention** replaces the target's equation. For example, `truth.do({"C": 2.0})` replaces $C=0.8A+\varepsilon_C$ with $C=2$. A keeps its usual equation and B still depends on A and C. Setting C is different from selecting observational samples where C happens to be near 2.

We collect observational samples and replicate measurements under `do(A)`, `do(B)`, and `do(C)`, each at −2 and +2. Two levels provide a clear response contrast. Under `do(B)`, neither A nor C should change on average, because B has no descendants.

`sample_data()` returns CORNETO `Data` and records the intervention on the target feature automatically. Each regime uses a separate random seed; these are independent samples, not paired counterfactuals.

In [ ]:
# We will generate 80 observational samples (no perturbations on the system)
n_obs = 80

# And 30 observations per intervention regime
n_per_intervention = 30

# Sample observations for A, B and C
observations = truth.sample_data(n_obs, rng=20, sample_ids=[f"obs_{i}" for i in range(n_obs)])
observations

In [ ]:
to_df(observations)

In [ ]:
# We will simulate interventions on every variable (A, B and C)
dataset = {"observation": observations}

for i, variable in enumerate(variables):
    for j, value in enumerate([-2.0, 2.0]):
        label = f"do({variable}={value:+.0f})"
        print("Performing intervention:", i, j, label)
        dataset[label] = truth.do({variable: value}).sample_data(
            n_per_intervention,
            rng=21 + 2*i + j, # generate a different seed per intervention
            sample_ids=[f"{label}_{k}" for k in range(n_per_intervention)],
            intervention_group=label
        )
dataset

In [ ]:
# Combine the observational and interventional Data objects into one Data object
data = Data.from_dict({sample_id: sample
                       for condition in dataset.values()
                       for sample_id, sample in condition.to_dict().items()})
data

In [ ]:
to_df(data)

In [ ]:
# Show the whole dataset grouped by obs / intervention
frames = []
for regime, part in dataset.items():
    frame = pd.DataFrame([
        {"sample_id": sample_id, **{f.id: f.value for f in sample.features}}
        for sample_id, sample in part.samples.items()
    ]).set_index("sample_id")
    frame["regime"] = regime
    frames.append(frame)
experiment = pd.concat(frames)
experiment.groupby("regime", sort=False)[variables].agg(["mean", "std"]).round(2)

In [ ]:
regime_means = experiment.groupby("regime", sort=False)[variables].mean()
ax = regime_means.plot.bar(figsize=(10, 4), color=["#0072B2", "#D55E00", "#009E73"])
ax.axhline(0, color="0.5", linewidth=0.8)
ax.set(ylabel="Mean measured value", xlabel="", title="Responses to hard interventions")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

## 4. Start discovery from a complete directed graph

We will now use this dataset as though we do not know the generating DAG. With only three variables, many samples and interventions on every variable, this is a relatively simple discovery setting under the method's assumptions.

We start with a complete directed graph on A, B and C, without self-loops, as our prior knowledge network. CORNETO will then search for a sparse DAG that fits the data.

In [ ]:
pkn = cn.Graph.from_tuples([(u, 1, v) for u in variables for v in variables if u != v])
# The tuple sign is a graph attribute; enforce_signs=False below leaves signs free.
pkn.plot()

## 5. Fit one model to observations and interventions

CORNETO's `LinearDAGDiscovery` method searches for a DAG within the supplied prior knowledge network that balances fit to observational and interventional data against the number of selected edges. We use absolute residual loss, scaled by each target's variation in the training data, plus an edge penalty controlled by `lambda_edges`. The `min_abs_coefficient` parameter sets the minimum magnitude of a selected coefficient in normalised units. These are modelling choices. We allow up to two parents per node and use zero intercepts to match the generating equations.

In [ ]:
method = LinearDAGDiscovery(
    lambda_edges=0.1, # cost (penalty) for selecting edges for the solution
    coefficient_bound=3.0, # cap on the coefficients to estimate [-3, 3]
    fit_intercept=False, # the generating SCM has zero intercepts
    max_parents=2, # we won't search beyond two parents per node.
    enforce_signs=False,
    coefficient_support="exact", # a selected edge has to have non zero coefficient
    min_abs_coefficient=0.01, # cap on min effect expected for selected edges (only if support = exact)
)

# Construct the CORNETO problem
problem = method.build(pkn, data)
problem

In [ ]:
# We can inspect all the variables automatically created for this problem
# Not all are relevant, some contain the important bits (like edges selected)
# while others are auxiliary variables required for the formulation
problem.expr

In [ ]:
# Run the HiGHS MILP solver to find an optimal solution
result = problem.solve(solver="highs", verbosity=1)
print("Solver status:", result.status)

In [ ]:
selected = np.asarray(problem.expr.edge_selected.value).reshape(-1) > 0.5
coefficients = np.asarray(problem.expr.edge_coefficient.value).reshape(-1)
edge_table = pd.DataFrame([
    {"source": next(iter(u)), "target": next(iter(v)),
     "selected": bool(selected[i]), "fitted coefficient": coefficients[i],
     "true coefficient": weights.get((next(iter(u)), next(iter(v))), 0.0)}
    for i, (u, v) in enumerate(pkn.E)
])
display(edge_table.round(3))
recovered = set(zip(edge_table.loc[selected, "source"], edge_table.loc[selected, "target"]))
print("Recovered generating edges:", recovered == set(weights))

styles = {
    i: ({"label": f"{coefficients[i]:+.2f}", "color": "#0072B2", "penwidth": "2.5"}
        if selected[i] else {"color": "#cccccc", "style": "dotted"})
    for i in range(pkn.num_edges)
}
pkn.plot(renderer="graphviz", graph_attr={"rankdir": "LR"}, custom_edge_attr=styles)

The solid arrows show selected edges; dotted arrows show rejected candidates. Compare the fitted coefficients with 0.8 (A → C), 1.4 (A → B), and 1.2 (C → B). Noise makes the estimates differ from the generating values.

In [ ]:
plot_dag_solution(method)

## 6. Predict new intervention levels

Now we will use the fitted model to make predictions for unseen interventions. The fitted graph and coefficients are fixed: `predict()` does not solve again. It uses hard-intervention values and propagates all values through the learned DAG.

**Note:** The fitted model does not simulate new noise, so compare its predictions with **regime means**, not individual noisy observations.

In [ ]:
test_parts = {
    target: truth.do({target: 1.0}).sample_data(
        300, rng=100 + i, sample_ids=[f"test_{target}_{j}" for j in range(300)],
        intervention_group=f"test_do_{target}_1",
    )
    for i, target in enumerate(variables)
}
rows = []
for target, test_data in test_parts.items():
    prediction = method.predict(test_data)
    for vertex in variables:
        measured = [next(f.value for f in s.features if f.id == vertex)
                    for s in test_data.samples.values()]
        predicted = [next(f.value for f in s.features if f.id == vertex)
                     for s in prediction.samples.values()]
        rows.append({"intervention": f"do({target}=1)", "variable": vertex,
                     "fixed by intervention": vertex == target,
                     "observed mean": np.mean(measured),
                     "predicted mean": np.mean(predicted)})
pd.DataFrame(rows).set_index(["intervention", "variable"]).round(3)

## 7. Explore

1. Double the noise standard deviations used to simulate data for A, B and C and rerun. Are the edges and coefficients stable?

2. Reduce `n_per_intervention`. How many replicates are sufficient in this simulation?

3. Increase `lambda_edges` enough to remove an edge. What fitting error does the simpler graph introduce?

4. Require each selected edge to have an absolute normalised coefficient of at least 1.5. What do you observe?

> Note: min_abs_coefficient sets the minimum strength of a selected edge in normalized units. For an edge $u \to v$, CORNETO requires $|\beta_{u\to v}|,\mathrm{scale}(u)/\mathrm{scale}(v) \geq \text{min\_abs\_coefficient}$. This makes the threshold comparable across variables with different units or spreads. Coefficients shown in the DAG remain in the original units.

5. Force each note to be at most 1 parent and plot the result

## 8. Can you guess the DAG?

The goal of this exercise is to propose a plausible DAG by combining your own knowledge with the data. The eight variables are continuous teaching scores centred around a population baseline of zero. Positive and negative values mean above and below that baseline; they are not literal counts, doses or probabilities. The table explains the variables without specifying any causal links.

| Variable | Positive values | Negative values |
| --- | --- | --- |
| `comorbidity` | Greater burden from coexisting conditions | Lower comorbidity burden |
| `severity` | More severe illness | Milder illness |
| `diagnosis` | Greater tendency to make a positive diagnostic decision | Lower tendency to make a positive diagnostic decision |
| `treatment` | More intensive disease-directed drug treatment | Less intensive treatment, not a negative drug dose |
| `inflammation` | More inflammatory activity | Less inflammatory activity |
| `side_effects` | Greater burden of side effects | Fewer or milder side effects |
| `function` | Better physical function | Poorer physical function |
| `recovery` | Better recovery status | Poorer recovery status |

The `diagnosis` score is a continuous assessment or decision score, not a yes/no disease label. Let us look at the data.

In [ ]:
import pandas as pd

df_dataset = pd.read_csv("data/simulation/experiment.csv")
df_dataset

In [ ]:
variables = df_dataset.columns[6:].tolist()
variables

In [ ]:
from corneto.graph import Graph

# Example of our prior believe for the DAG.
# NOTE: this is completely wrong, try to define a better one.

edges = [
    ("inflammation", 1, "comorbidity"),
    ("inflammation", 1, "side_effects"),
    ("function", 1, "side_effects"),
    ("recovery", 1, "diagnosis"),
    ("inflammation", 1, "treatment"),
    ("side_effects", 1, "severity"),
    ("function", 1, "inflammation"),
    ("recovery", 1, "function"),
    ("diagnosis", 1, "severity"),
    ("treatment", 1, "severity")
]

pkn = Graph.from_tuples(edges)

pkn.V

In [ ]:
# Just to make plots prettier
node_style = {
    "shape": "box",
    "style": "rounded,filled",
    "fillcolor": "#edf3f8",
    "color": "#43566b",
    "fontcolor": "#213247",
    "fontname": "Helvetica",
    "fontsize": "10",
    "margin": "0.16,0.09",
}

pkn.plot(
    renderer="graphviz",
    graph_attr={"layout": "circo", "bgcolor": "transparent"},
    node_attr=node_style,
    edge_attr={"color": "#aab7c4", "arrowsize": "0.5", "penwidth": "0.8"},
)

In [ ]:
from utils import (
    from_df, plot_dag_solution, evaluate_dag,
    plot_dag_errors, plot_dag_predictions,
)

# Convert the dataframe to CORNETO format
data = from_df(df_dataset, sample_id_column="sample_id")
data

In [ ]:
method = LinearDAGDiscovery(
    # Increasing the edge penalty reduces the risk of false positives
    lambda_edges=0.001,
    # Average the fit loss per sample so the edge penalty is independent of sample count.
    sample_weights={sample_id: 1.0 / len(data.samples) for sample_id in data.samples},
    # Upper/lower bound on coefficient magnitude
    coefficient_bound=3.0,
    # Max num of parents per node
    max_parents=3,
    # No bias term
    fit_intercept=False,
    # Do not constrain fitted coefficients to the PKN interaction signs
    enforce_signs=False,
    coefficient_support="exact",
    min_abs_coefficient=0.01,
)
method.fit(pkn, data, solver="HIGHS", max_seconds=300, verbosity=1)

In [ ]:
# Plot only the structure
pkn.edge_subgraph(np.flatnonzero(method.problem.expr.edge_selected.value > 0.5)).plot()

In [ ]:
plot_dag_solution(method)

In [ ]:
plot_dag_predictions(method, data);

## 9. Start from a complete (or almost complete) graph

Now assume that we do not know the DAG. We will start with a complete directed graph as an uninformative prior. To reduce the search space, comment out any interactions that you suspect are absent.

In [ ]:
# Build the prior knowledge graph from all candidate interactions
# Comment out individual lines to exclude interactions you suspect are absent

edges = [
    ("comorbidity", 1, "severity"),
    ("comorbidity", 1, "diagnosis"),
    ("comorbidity", 1, "treatment"),
    ("comorbidity", 1, "inflammation"),
    ("comorbidity", 1, "side_effects"),
    ("comorbidity", 1, "function"),
    ("comorbidity", 1, "recovery"),

    ("severity", 1, "comorbidity"),
    ("severity", 1, "diagnosis"),
    ("severity", 1, "treatment"),
    ("severity", 1, "inflammation"),
    ("severity", 1, "side_effects"),
    ("severity", 1, "function"),
    ("severity", 1, "recovery"),

    ("diagnosis", 1, "comorbidity"),
    ("diagnosis", 1, "severity"),
    ("diagnosis", 1, "treatment"),
    ("diagnosis", 1, "inflammation"),
    ("diagnosis", 1, "side_effects"),
    ("diagnosis", 1, "function"),
    ("diagnosis", 1, "recovery"),

    ("treatment", 1, "comorbidity"),
    ("treatment", 1, "severity"),
    ("treatment", 1, "diagnosis"),
    ("treatment", 1, "inflammation"),
    ("treatment", 1, "side_effects"),
    ("treatment", 1, "function"),
    ("treatment", 1, "recovery"),

    ("inflammation", 1, "comorbidity"),
    ("inflammation", 1, "severity"),
    ("inflammation", 1, "diagnosis"),
    ("inflammation", 1, "treatment"),
    ("inflammation", 1, "side_effects"),
    ("inflammation", 1, "function"),
    ("inflammation", 1, "recovery"),

    ("side_effects", 1, "comorbidity"),
    ("side_effects", 1, "severity"),
    ("side_effects", 1, "diagnosis"),
    ("side_effects", 1, "treatment"),
    ("side_effects", 1, "inflammation"),
    ("side_effects", 1, "function"),
    ("side_effects", 1, "recovery"),

    ("function", 1, "comorbidity"),
    ("function", 1, "severity"),
    ("function", 1, "diagnosis"),
    ("function", 1, "treatment"),
    ("function", 1, "inflammation"),
    ("function", 1, "side_effects"),
    ("function", 1, "recovery"),

    ("recovery", 1, "comorbidity"),
    ("recovery", 1, "severity"),
    ("recovery", 1, "diagnosis"),
    ("recovery", 1, "treatment"),
    ("recovery", 1, "inflammation"),
    ("recovery", 1, "side_effects"),
    ("recovery", 1, "function"),
]

pkn = Graph.from_tuples(edges)

pkn.shape

In [ ]:
# Run here the method using this larger PKN after you remove some of the less plausible edges

## 10. Train on a smaller dataset

We will now use a smaller dataset with interventions on four of the eight variables: comorbidity, function, inflammation, and treatment. This reflects the fact that some variables, such as side effects, may be difficult to intervene on directly. We will see what can be inferred from the available interventions.


In [ ]:
# Load the subset using the sample IDs and intervention labels in the CSV.
small_dataset = pd.read_csv("data/simulation/experiment_subset_comorbidity.csv", keep_default_na=False)
small_data = from_df(small_dataset, sample_id_column="sample_id")
small_data

In [ ]:
method = LinearDAGDiscovery(
    # Increasing the edge penalty reduces the risk of false positives
    lambda_edges=0.01,
    # Average the fit loss per sample so the edge penalty is independent of sample count.
    sample_weights={sample_id: 1.0 / len(small_data.samples) for sample_id in small_data.samples},
    # Upper/lower bound on coefficient magnitude
    coefficient_bound=3.0,
    # Max num of parents per node
    max_parents=3,
    # No bias term
    fit_intercept=False,
    # Do not constrain fitted coefficients to the PKN interaction signs
    enforce_signs=False,
    coefficient_support="exact",
    min_abs_coefficient=0.01,
)
method.fit(pkn, small_data, solver="HIGHS", max_seconds=300, verbosity=1)

In [ ]:
plot_dag_solution(method)

In [ ]:
small_errors = evaluate_dag(method, small_data)
display(small_errors["summary"].round(3).to_frame("value"))
display(small_errors["by_variable"].round(3))

In [ ]:
# Evaluate the model fitted on the subset against all available regimes
plot_dag_predictions(method, data);

In [ ]:
# Compare with its predictions on the training subset
plot_dag_predictions(method, small_data);